# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/19-PythonFlaskSQLiteCRUD.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 19 - Flask + SQLite ile Web CRUD Uygulaması

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bu derste önceki derslerde öğrendiğimiz:

- Flask,
- HTML,
- Jinja,
- form işlemleri,
- SQLite,
- SQL,
- CRUD

konularını tek bir web uygulamasında birleştireceğiz.

Geliştireceğimiz proje bir **Öğrenci Kayıt Web Uygulaması** olacaktır.

> **Çalışma ortamı notu:** Flask uygulamasını Visual Studio, VS Code veya yerel Python ortamında çalıştırmanız önerilir. Google Colab bu notebook'u okumak ve kodları incelemek için kullanılabilir.

Bu dersin sonunda öğrencinin:

- Flask uygulamasını SQLite'a bağlayabilmesi,
- veritabanı tablosu oluşturabilmesi,
- kayıt listeleyebilmesi,
- yeni kayıt ekleyebilmesi,
- kayıt düzenleyebilmesi,
- kayıt silebilmesi,
- isimle arama yapabilmesi,
- HTML formlarını doğrulayabilmesi,
- flash mesajı gösterebilmesi,
- ID tabanlı dinamik route kullanabilmesi,
- Jinja ile verileri tabloya aktarabilmesi,
- CRUD web uygulamasının tam çalışma akışını kavrayabilmesi

hedeflenmektedir.


# 1. Proje Senaryosu

Web uygulamamızda öğrenciler için şu bilgileri tutacağız:

- ID
- öğrenci adı
- sınıf
- Python puanı
- Matematik puanı

Uygulamada şu sayfalar bulunacak:

```text
/                   Ana sayfa
/ogrenciler         Öğrenci listesi
/ogrenci-ekle       Yeni öğrenci ekleme
/ogrenci/<id>        Öğrenci detay
/ogrenci/<id>/duzenle
/ogrenci/<id>/sil
/ara
```

Bu yapı gerçek bir CRUD web uygulamasının temelini oluşturur.


# 2. CRUD Akışını Hatırlayalım

| CRUD | SQL | Web İşlemi |
|---|---|---|
| Create | INSERT | Öğrenci ekle |
| Read | SELECT | Öğrencileri listele |
| Update | UPDATE | Öğrenciyi düzenle |
| Delete | DELETE | Öğrenciyi sil |

Web arayüzü değişse de SQLite tarafındaki SQL mantığı aynıdır.


# 3. Proje Klasör Yapısı

Bu derste aşağıdaki yapıyı kullanacağız:

```text
ogrenci_web/
│
├── app.py
├── ogrenciler.db
│
├── templates/
│   ├── base.html
│   ├── index.html
│   ├── ogrenciler.html
│   ├── ogrenci_ekle.html
│   ├── ogrenci_duzenle.html
│   ├── ogrenci_detay.html
│   └── 404.html
│
└── static/
    └── css/
        └── style.css
```

İlk hedefimiz çalışan ve anlaşılır bir yapı kurmaktır.


# 4. Sanal Ortam ve Flask Kurulumu

Proje klasöründe:

```text
python -m venv .venv
```

Windows:

```text
.venv\Scripts\activate
```

Flask kurulumu:

```text
pip install Flask
```

Uygulama:

```text
flask --app app run --debug
```

ile geliştirilebilir.

> Debug modu yalnızca geliştirme ortamında kullanılmalıdır.


# 5. İlk `app.py` İçe Aktarmaları

In [ ]:
from flask import (
    Flask,
    render_template,
    request,
    redirect,
    url_for,
    flash,
    abort,
    g
)

import sqlite3


Bu derste:

- `render_template` → HTML göstermek,
- `request` → form ve query verisi almak,
- `redirect` → yönlendirmek,
- `url_for` → endpoint URL'si oluşturmak,
- `flash` → kullanıcı mesajı göstermek,
- `abort` → HTTP hata cevabı üretmek,
- `g` → istek boyunca veritabanı bağlantısını tutmak

için kullanılacaktır.


# 6. Flask Uygulamasını Oluşturmak

In [ ]:
app = Flask(__name__)

app.secret_key = "gelistirme-icin-ornek-anahtar"


Flash mesajlarının ve oturum tabanlı bazı yapıların çalışabilmesi için secret key gerekir.

Gerçek projelerde secret key:

- tahmin edilemez olmalı,
- kaynak kod içinde açık tutulmamalı,
- ortam değişkeni gibi güvenli bir yerden alınmalıdır.

Bu derste mekanizmayı öğrenmek için basit bir örnek kullanıyoruz.


# 7. Veritabanı Dosyası

In [ ]:
DATABASE = "ogrenciler.db"


# 8. Veritabanı Bağlantısı İçin `get_db()`

Her HTTP isteği sırasında gerektiğinde bir SQLite bağlantısı oluşturabiliriz.


In [ ]:
def get_db():
    if "db" not in g:
        g.db = sqlite3.connect(DATABASE)
        g.db.row_factory = sqlite3.Row

    return g.db


`sqlite3.Row` kullandığımız için sorgu sonuçlarına:

```python
ogrenci["Isim"]
```

gibi sütun adıyla erişebiliriz.


# 9. İstek Sonunda Bağlantıyı Kapatmak

In [ ]:
@app.teardown_appcontext
def close_db(exception=None):
    db = g.pop("db", None)

    if db is not None:
        db.close()


Bu yapı, istek tamamlandığında açılmış bağlantının kapatılmasını sağlar.


# 10. Veritabanı Tablosunu Hazırlamak

In [ ]:
def veritabani_hazirla():
    db = sqlite3.connect(DATABASE)

    db.execute("""
    CREATE TABLE IF NOT EXISTS Ogrenciler (
        Id INTEGER PRIMARY KEY AUTOINCREMENT,
        Isim TEXT NOT NULL,
        Sinif INTEGER NOT NULL,
        PythonPuani INTEGER NOT NULL,
        MatematikPuani INTEGER NOT NULL
    )
    """)

    db.commit()
    db.close()


Uygulama başlamadan önce bir kez:

```python
veritabani_hazirla()
```

çağırabiliriz.


# 11. Örnek Kayıtlar Eklemek

İlk test için birkaç kayıt ekleyebiliriz.

Bu kodu yalnızca bir kez çalıştırmak gerekir.


In [ ]:
def ornek_kayitlar():
    db = sqlite3.connect(DATABASE)

    adet = db.execute(
        "SELECT COUNT(*) FROM Ogrenciler"
    ).fetchone()[0]

    if adet == 0:
        db.executemany("""
        INSERT INTO Ogrenciler (
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani
        )
        VALUES (?, ?, ?, ?)
        """, [
            ("Ali", 8, 90, 85),
            ("Ayşe", 8, 95, 92),
            ("Mehmet", 7, 72, 75),
            ("Zeynep", 8, 88, 94)
        ])

        db.commit()

    db.close()


Bu kontrol sayesinde uygulama her başladığında aynı örnek öğrenciler tekrar tekrar eklenmez.


# 12. Uygulamayı Hazırlamak

In [ ]:
veritabani_hazirla()
ornek_kayitlar()


# 13. Ana Sayfa Route'u

In [ ]:
@app.route("/")
def ana_sayfa():
    return render_template("index.html")


# 14. `base.html`

Ortak HTML yapısını tek bir dosyada tutalım.

```html
<!DOCTYPE html>
<html lang="tr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>
        {% block title %}
        Öğrenci Sistemi
        {% endblock %}
    </title>

    <link
        rel="stylesheet"
        href="{{ url_for('static', filename='css/style.css') }}"
    >
</head>
<body>

<header>
    <h1>Niyazi Sayın BİLSEM</h1>

    <nav>
        <a href="{{ url_for('ana_sayfa') }}">Ana Sayfa</a>
        <a href="{{ url_for('ogrenci_listesi') }}">Öğrenciler</a>
        <a href="{{ url_for('ogrenci_ekle') }}">Yeni Öğrenci</a>
    </nav>
</header>

<main>

    {% with mesajlar = get_flashed_messages() %}
        {% if mesajlar %}
            {% for mesaj in mesajlar %}
                <div class="mesaj">
                    {{ mesaj }}
                </div>
            {% endfor %}
        {% endif %}
    {% endwith %}

    {% block content %}
    {% endblock %}

</main>

</body>
</html>
```


# 15. `index.html`

```html
{% extends "base.html" %}

{% block title %}
Ana Sayfa
{% endblock %}

{% block content %}

<h2>Öğrenci Kayıt Sistemi</h2>

<p>
    Flask ve SQLite ile hazırlanmış örnek web uygulamasına hoş geldiniz.
</p>

<p>
    Bu uygulamada öğrenci ekleyebilir, listeleyebilir,
    düzenleyebilir ve silebilirsiniz.
</p>

{% endblock %}
```


# 16. Read: Öğrencileri Veritabanından Okumak

İlk CRUD işlemimiz kayıtları listelemek olacaktır.


In [ ]:
@app.route("/ogrenciler")
def ogrenci_listesi():
    db = get_db()

    ogrenciler = db.execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    ORDER BY Id DESC
    """).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler
    )


# 17. `ogrenciler.html`

```html
{% extends "base.html" %}

{% block title %}
Öğrenciler
{% endblock %}

{% block content %}

<h2>Öğrenciler</h2>

<table>
    <thead>
        <tr>
            <th>ID</th>
            <th>İsim</th>
            <th>Sınıf</th>
            <th>Python</th>
            <th>Matematik</th>
            <th>Ortalama</th>
            <th>İşlemler</th>
        </tr>
    </thead>

    <tbody>

        {% for ogrenci in ogrenciler %}
        <tr>
            <td>{{ ogrenci["Id"] }}</td>
            <td>{{ ogrenci["Isim"] }}</td>
            <td>{{ ogrenci["Sinif"] }}</td>
            <td>{{ ogrenci["PythonPuani"] }}</td>
            <td>{{ ogrenci["MatematikPuani"] }}</td>
            <td>{{ "%.2f"|format(ogrenci["Ortalama"]) }}</td>

            <td>
                <a href="{{ url_for('ogrenci_detay', ogrenci_id=ogrenci['Id']) }}">
                    Detay
                </a>

                <a href="{{ url_for('ogrenci_duzenle', ogrenci_id=ogrenci['Id']) }}">
                    Düzenle
                </a>
            </td>
        </tr>
        {% endfor %}

    </tbody>
</table>

{% endblock %}
```


# 18. Jinja ile Boş Liste Kontrolü

Hiç öğrenci yoksa boş tablo yerine açıklama gösterebiliriz.

```html
{% if ogrenciler %}

    <table>
        ...
    </table>

{% else %}

    <p>Henüz öğrenci kaydı yok.</p>

{% endif %}
```


# 19. Bir Öğrenciyi ID ile Getiren Yardımcı Fonksiyon

Birden fazla route aynı öğrenci sorgusuna ihtiyaç duyacaktır.

Tekrarı azaltmak için fonksiyon yazalım.


In [ ]:
def ogrenci_getir(ogrenci_id):
    db = get_db()

    ogrenci = db.execute("""
    SELECT *
    FROM Ogrenciler
    WHERE Id = ?
    """, (ogrenci_id,)).fetchone()

    if ogrenci is None:
        abort(404)

    return ogrenci


Bu fonksiyon:

- ID'ye göre kayıt arar,
- kayıt yoksa 404 üretir,
- varsa öğrenci kaydını döndürür.


# 20. Öğrenci Detay Route'u

In [ ]:
@app.route("/ogrenci/<int:ogrenci_id>")
def ogrenci_detay(ogrenci_id):
    ogrenci = ogrenci_getir(ogrenci_id)

    ortalama = (
        ogrenci["PythonPuani"] +
        ogrenci["MatematikPuani"]
    ) / 2

    return render_template(
        "ogrenci_detay.html",
        ogrenci=ogrenci,
        ortalama=ortalama
    )


# 21. `ogrenci_detay.html`

```html
{% extends "base.html" %}

{% block title %}
Öğrenci Detayı
{% endblock %}

{% block content %}

<h2>{{ ogrenci["Isim"] }}</h2>

<p>ID: {{ ogrenci["Id"] }}</p>
<p>Sınıf: {{ ogrenci["Sinif"] }}</p>
<p>Python: {{ ogrenci["PythonPuani"] }}</p>
<p>Matematik: {{ ogrenci["MatematikPuani"] }}</p>
<p>Ortalama: {{ "%.2f"|format(ortalama) }}</p>

<a href="{{ url_for('ogrenci_duzenle', ogrenci_id=ogrenci['Id']) }}">
    Düzenle
</a>

{% endblock %}
```


# 22. Form Doğrulama Fonksiyonu

Ekleme ve düzenleme ekranlarında aynı kontrolleri kullanacağız.


In [ ]:
def form_dogrula(form):
    isim = form.get("isim", "").strip()
    sinif_text = form.get("sinif", "").strip()
    python_text = form.get("python", "").strip()
    matematik_text = form.get("matematik", "").strip()

    if not isim:
        return None, "Öğrenci adı boş bırakılamaz."

    try:
        sinif = int(sinif_text)
        python_puani = int(python_text)
        matematik_puani = int(matematik_text)

    except ValueError:
        return None, "Sınıf ve puan alanları sayısal olmalıdır."

    if sinif < 1 or sinif > 12:
        return None, "Sınıf 1-12 arasında olmalıdır."

    if not 0 <= python_puani <= 100:
        return None, "Python puanı 0-100 arasında olmalıdır."

    if not 0 <= matematik_puani <= 100:
        return None, "Matematik puanı 0-100 arasında olmalıdır."

    veri = (
        isim,
        sinif,
        python_puani,
        matematik_puani
    )

    return veri, None


# 23. Create: Öğrenci Ekleme Route'u

In [ ]:
@app.route("/ogrenci-ekle", methods=["GET", "POST"])
def ogrenci_ekle():
    if request.method == "POST":
        veri, hata = form_dogrula(request.form)

        if hata:
            flash(hata)

        else:
            db = get_db()

            db.execute("""
            INSERT INTO Ogrenciler (
                Isim,
                Sinif,
                PythonPuani,
                MatematikPuani
            )
            VALUES (?, ?, ?, ?)
            """, veri)

            db.commit()

            flash("Öğrenci başarıyla eklendi.")

            return redirect(
                url_for("ogrenci_listesi")
            )

    return render_template(
        "ogrenci_ekle.html"
    )


# 24. `ogrenci_ekle.html`

```html
{% extends "base.html" %}

{% block title %}
Öğrenci Ekle
{% endblock %}

{% block content %}

<h2>Yeni Öğrenci</h2>

<form method="post">

    <p>
        <label>Öğrenci Adı</label>
        <input
            type="text"
            name="isim"
            value="{{ request.form.get('isim', '') }}"
            required
        >
    </p>

    <p>
        <label>Sınıf</label>
        <input
            type="number"
            name="sinif"
            min="1"
            max="12"
            value="{{ request.form.get('sinif', '') }}"
            required
        >
    </p>

    <p>
        <label>Python Puanı</label>
        <input
            type="number"
            name="python"
            min="0"
            max="100"
            value="{{ request.form.get('python', '') }}"
            required
        >
    </p>

    <p>
        <label>Matematik Puanı</label>
        <input
            type="number"
            name="matematik"
            min="0"
            max="100"
            value="{{ request.form.get('matematik', '') }}"
            required
        >
    </p>

    <button type="submit">
        Kaydet
    </button>

</form>

{% endblock %}
```


# 25. Neden Hem HTML Hem Flask Doğrulaması?

HTML tarafındaki:

```html
required
min
max
```

kullanıcı deneyimini iyileştirir.

Ancak bunlar tek başına güvenli değildir.

Sunucu tarafındaki Python kodu da veriyi mutlaka kontrol etmelidir.

Doğru yaklaşım:

**Tarayıcı Doğrulaması + Sunucu Doğrulaması**


# 26. POST Sonrası Redirect

Başarılı kayıt sonrası:

```python
return redirect(
    url_for("ogrenci_listesi")
)
```

kullanıyoruz.

Böylece kullanıcı yeniden yükleme yaptığında aynı formun istemeden tekrar gönderilmesi riski azaltılır.


# 27. Update: Düzenleme Route'u

Önce mevcut kaydı getiriyoruz.

GET isteğinde formu dolduruyoruz.

POST isteğinde güncelliyoruz.


In [ ]:
@app.route(
    "/ogrenci/<int:ogrenci_id>/duzenle",
    methods=["GET", "POST"]
)
def ogrenci_duzenle(ogrenci_id):
    ogrenci = ogrenci_getir(ogrenci_id)

    if request.method == "POST":
        veri, hata = form_dogrula(request.form)

        if hata:
            flash(hata)

        else:
            isim, sinif, python_puani, matematik_puani = veri

            db = get_db()

            db.execute("""
            UPDATE Ogrenciler
            SET
                Isim = ?,
                Sinif = ?,
                PythonPuani = ?,
                MatematikPuani = ?
            WHERE Id = ?
            """, (
                isim,
                sinif,
                python_puani,
                matematik_puani,
                ogrenci_id
            ))

            db.commit()

            flash("Öğrenci kaydı güncellendi.")

            return redirect(
                url_for(
                    "ogrenci_detay",
                    ogrenci_id=ogrenci_id
                )
            )

    return render_template(
        "ogrenci_duzenle.html",
        ogrenci=ogrenci
    )


# 28. `ogrenci_duzenle.html`

```html
{% extends "base.html" %}

{% block title %}
Öğrenci Düzenle
{% endblock %}

{% block content %}

<h2>Öğrenci Düzenle</h2>

<form method="post">

    <p>
        <label>Öğrenci Adı</label>
        <input
            type="text"
            name="isim"
            value="{{ request.form.get('isim', ogrenci['Isim']) }}"
            required
        >
    </p>

    <p>
        <label>Sınıf</label>
        <input
            type="number"
            name="sinif"
            value="{{ request.form.get('sinif', ogrenci['Sinif']) }}"
            required
        >
    </p>

    <p>
        <label>Python Puanı</label>
        <input
            type="number"
            name="python"
            value="{{ request.form.get('python', ogrenci['PythonPuani']) }}"
            required
        >
    </p>

    <p>
        <label>Matematik Puanı</label>
        <input
            type="number"
            name="matematik"
            value="{{ request.form.get('matematik', ogrenci['MatematikPuani']) }}"
            required
        >
    </p>

    <button type="submit">
        Güncelle
    </button>

</form>

{% endblock %}
```


# 29. Delete: Silme Route'u

Silme işlemi veri değiştirdiği için POST kullanacağız.


In [ ]:
@app.post("/ogrenci/<int:ogrenci_id>/sil")
def ogrenci_sil(ogrenci_id):
    ogrenci_getir(ogrenci_id)

    db = get_db()

    db.execute(
        "DELETE FROM Ogrenciler WHERE Id = ?",
        (ogrenci_id,)
    )

    db.commit()

    flash("Öğrenci kaydı silindi.")

    return redirect(
        url_for("ogrenci_listesi")
    )


# 30. Neden Silme İçin GET Kullanmıyoruz?

Şu yapı önerilmez:

```text
GET /ogrenci/5/sil
```

GET işlemleri normalde veri okumak için düşünülmelidir.

Silme gibi sunucudaki veriyi değiştiren işlemlerde POST kullanmak daha doğru bir web tasarımıdır.


# 31. Silme Formu

`ogrenci_detay.html` veya liste tablosunda:

```html
<form
    method="post"
    action="{{ url_for('ogrenci_sil', ogrenci_id=ogrenci['Id']) }}"
    onsubmit="return confirm('Bu kayıt silinsin mi?');"
>

    <button type="submit">
        Sil
    </button>

</form>
```

Burada tarayıcı tarafında basit bir silme onayı kullanılmıştır.


# 32. Search: İsimle Arama Route'u

In [ ]:
@app.route("/ara")
def ogrenci_ara():
    kelime = request.args.get(
        "q",
        ""
    ).strip()

    db = get_db()

    ogrenciler = db.execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    WHERE Isim LIKE ?
    ORDER BY Isim
    """, (
        f"%{kelime}%",
    )).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler,
        arama=kelime
    )


# 33. Arama Formu

`ogrenciler.html` sayfasına:

```html
<form
    method="get"
    action="{{ url_for('ogrenci_ara') }}"
>

    <input
        type="text"
        name="q"
        value="{{ arama|default('') }}"
        placeholder="Öğrenci adı ara"
    >

    <button type="submit">
        Ara
    </button>

</form>
```

Aramada GET kullanmamız mantıklıdır çünkü arama sunucu verisini değiştirmez.


# 34. Filtreleme: Sınıfa Göre Öğrenci Listesi

In [ ]:
@app.route("/sinif/<int:sinif>")
def sinif_ogrencileri(sinif):
    db = get_db()

    ogrenciler = db.execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    WHERE Sinif = ?
    ORDER BY Isim
    """, (sinif,)).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler
    )


# 35. Sıralama Route'u

Örneğin ortalamaya göre büyükten küçüğe:


In [ ]:
@app.route("/ogrenciler/basariya-gore")
def basariya_gore():
    db = get_db()

    ogrenciler = db.execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    ORDER BY Ortalama DESC
    """).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler
    )


# 36. İstatistik Sayfası

Veritabanından özet bilgiler hesaplayabiliriz.


In [ ]:
@app.route("/istatistik")
def istatistik():
    db = get_db()

    sonuc = db.execute("""
    SELECT
        COUNT(*) AS OgrenciSayisi,
        AVG((PythonPuani + MatematikPuani) / 2.0) AS GenelOrtalama,
        MAX((PythonPuani + MatematikPuani) / 2.0) AS EnYuksek,
        MIN((PythonPuani + MatematikPuani) / 2.0) AS EnDusuk
    FROM Ogrenciler
    """).fetchone()

    return render_template(
        "istatistik.html",
        sonuc=sonuc
    )


# 37. `istatistik.html`

```html
{% extends "base.html" %}

{% block content %}

<h2>İstatistikler</h2>

<p>
    Toplam öğrenci:
    {{ sonuc["OgrenciSayisi"] }}
</p>

<p>
    Genel ortalama:
    {{ "%.2f"|format(sonuc["GenelOrtalama"] or 0) }}
</p>

<p>
    En yüksek:
    {{ "%.2f"|format(sonuc["EnYuksek"] or 0) }}
</p>

<p>
    En düşük:
    {{ "%.2f"|format(sonuc["EnDusuk"] or 0) }}
</p>

{% endblock %}
```


# 38. Sınıflara Göre Ortalama

`GROUP BY` web uygulamasında da aynen kullanılabilir.


In [ ]:
@app.route("/sinif-istatistik")
def sinif_istatistik():
    db = get_db()

    sonuclar = db.execute("""
    SELECT
        Sinif,
        COUNT(*) AS OgrenciSayisi,
        AVG(
            (PythonPuani + MatematikPuani) / 2.0
        ) AS Ortalama
    FROM Ogrenciler
    GROUP BY Sinif
    ORDER BY Sinif
    """).fetchall()

    return render_template(
        "sinif_istatistik.html",
        sonuclar=sonuclar
    )


# 39. 404 Hata Sayfası

Kayıt bulunamazsa `ogrenci_getir()` fonksiyonu:

```python
abort(404)
```

çağırır.

Özel hata sayfası oluşturalım.


In [ ]:
@app.errorhandler(404)
def sayfa_bulunamadi(hata):
    return render_template(
        "404.html"
    ), 404


# 40. `404.html`

```html
{% extends "base.html" %}

{% block content %}

<h2>Sayfa Bulunamadı</h2>

<p>
    Aradığınız sayfa veya kayıt bulunamadı.
</p>

<a href="{{ url_for('ana_sayfa') }}">
    Ana sayfaya dön
</a>

{% endblock %}
```


# 41. CSS Dosyası

`static/css/style.css`:

```css
body {
    font-family: Arial, sans-serif;
    margin: 0;
    background: #f5f5f5;
}

header,
main {
    max-width: 1000px;
    margin: auto;
    padding: 20px;
}

nav a {
    margin-right: 15px;
}

table {
    width: 100%;
    border-collapse: collapse;
    background: white;
}

th,
td {
    border: 1px solid #ccc;
    padding: 10px;
    text-align: left;
}

form p {
    margin-bottom: 12px;
}

input {
    padding: 8px;
}

button {
    padding: 8px 14px;
}

.mesaj {
    padding: 10px;
    margin-bottom: 10px;
    background: white;
    border: 1px solid #ccc;
}
```

Burada amacımız ayrıntılı CSS tasarımı değil, web uygulamasının okunabilir hale gelmesidir.


# 42. Parametreli SQL Neden Önemli?

Yanlış:

```python
sql = "SELECT * FROM Ogrenciler WHERE Isim = '" + isim + "'"
```

Doğru:

```python
db.execute(
    "SELECT * FROM Ogrenciler WHERE Isim = ?",
    (isim,)
)
```

Kullanıcı verisini SQL cümlesine string birleştirmeyle eklemiyoruz.


# 43. Kullanıcı Verisini HTML'e Nasıl Yazıyoruz?

Template içinde:

```html
{{ ogrenci["Isim"] }}
```

kullanıyoruz.

Jinja, HTML template'lerinde normal değişken çıktılarında escaping uygular.

Kullanıcıdan gelen metinleri gereksiz yere:

```html
{{ veri|safe }}
```

şeklinde göstermemeliyiz.


# 44. Veritabanı Hatalarını Yönetmek

Örneğin öğrenci numarası gibi `UNIQUE` alan eklediğimizde tekrar eden değer hata oluşturabilir.

```python
try:
    db.execute(...)
    db.commit()

except sqlite3.IntegrityError:
    flash("Bu öğrenci numarası zaten kayıtlı.")
```

Veritabanı kurallarını kullanıcıya anlaşılır mesajlarla aktarmak gerekir.


# 45. Öğrenci Numarası Eklemek

Tablomuzu geliştirirken:

```sql
OgrenciNo INTEGER UNIQUE NOT NULL
```

alanı ekleyebiliriz.

Bu durumda:

- ekleme formu,
- düzenleme formu,
- INSERT,
- UPDATE,
- HTML tablo

da yeni alana göre güncellenmelidir.


# 46. POST İşlemlerinde Güvenlik Düşüncesi

Bir web uygulamasında yalnızca SQL Injection değil, form üzerinden istenmeyen istekler de önemlidir.

Daha ileri Flask projelerinde:

- CSRF koruması,
- kullanıcı oturumları,
- yetkilendirme,
- parola hashleme

gibi konular ele alınmalıdır.

Bu dersin amacı temel CRUD mantığıdır; ancak güvenlik düşüncesini en baştan kazanmak önemlidir.


# 47. Parola Saklama Konusu

İleride kullanıcı giriş sistemi yaptığımızda parolalar veritabanında düz metin olarak tutulmamalıdır.

Flask ekosisteminde ve Werkzeug araçlarında parola hashleme için uygun fonksiyonlar vardır.

Bu CRUD dersimizde kullanıcı hesabı bulunmadığı için parola sistemi geliştirmiyoruz.


# 48. Tam `app.py`

Şimdi ana Python dosyamızın temel parçalarını tek bir örnekte birleştirelim.


In [ ]:
from flask import (
    Flask,
    render_template,
    request,
    redirect,
    url_for,
    flash,
    abort,
    g
)
import sqlite3

app = Flask(__name__)
app.secret_key = "gelistirme-icin-ornek-anahtar"

DATABASE = "ogrenciler.db"

def get_db():
    if "db" not in g:
        g.db = sqlite3.connect(DATABASE)
        g.db.row_factory = sqlite3.Row
    return g.db

@app.teardown_appcontext
def close_db(exception=None):
    db = g.pop("db", None)
    if db is not None:
        db.close()

def veritabani_hazirla():
    db = sqlite3.connect(DATABASE)
    db.execute("""
    CREATE TABLE IF NOT EXISTS Ogrenciler (
        Id INTEGER PRIMARY KEY AUTOINCREMENT,
        Isim TEXT NOT NULL,
        Sinif INTEGER NOT NULL,
        PythonPuani INTEGER NOT NULL,
        MatematikPuani INTEGER NOT NULL
    )
    """)
    db.commit()
    db.close()

def ogrenci_getir(ogrenci_id):
    ogrenci = get_db().execute(
        "SELECT * FROM Ogrenciler WHERE Id = ?",
        (ogrenci_id,)
    ).fetchone()

    if ogrenci is None:
        abort(404)

    return ogrenci

def form_dogrula(form):
    isim = form.get("isim", "").strip()

    try:
        sinif = int(form.get("sinif", ""))
        python_puani = int(form.get("python", ""))
        matematik_puani = int(form.get("matematik", ""))
    except ValueError:
        return None, "Sınıf ve puanlar sayısal olmalıdır."

    if not isim:
        return None, "Öğrenci adı boş bırakılamaz."

    if not 1 <= sinif <= 12:
        return None, "Sınıf 1-12 arasında olmalıdır."

    if not 0 <= python_puani <= 100:
        return None, "Python puanı 0-100 arasında olmalıdır."

    if not 0 <= matematik_puani <= 100:
        return None, "Matematik puanı 0-100 arasında olmalıdır."

    return (
        isim,
        sinif,
        python_puani,
        matematik_puani
    ), None

@app.route("/")
def ana_sayfa():
    return render_template("index.html")

@app.route("/ogrenciler")
def ogrenci_listesi():
    ogrenciler = get_db().execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    ORDER BY Id DESC
    """).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler
    )

@app.route("/ogrenci/<int:ogrenci_id>")
def ogrenci_detay(ogrenci_id):
    ogrenci = ogrenci_getir(ogrenci_id)

    ortalama = (
        ogrenci["PythonPuani"] +
        ogrenci["MatematikPuani"]
    ) / 2

    return render_template(
        "ogrenci_detay.html",
        ogrenci=ogrenci,
        ortalama=ortalama
    )

@app.route("/ogrenci-ekle", methods=["GET", "POST"])
def ogrenci_ekle():
    if request.method == "POST":
        veri, hata = form_dogrula(request.form)

        if hata:
            flash(hata)
        else:
            db = get_db()

            db.execute("""
            INSERT INTO Ogrenciler (
                Isim,
                Sinif,
                PythonPuani,
                MatematikPuani
            )
            VALUES (?, ?, ?, ?)
            """, veri)

            db.commit()

            flash("Öğrenci başarıyla eklendi.")

            return redirect(
                url_for("ogrenci_listesi")
            )

    return render_template("ogrenci_ekle.html")

@app.route(
    "/ogrenci/<int:ogrenci_id>/duzenle",
    methods=["GET", "POST"]
)
def ogrenci_duzenle(ogrenci_id):
    ogrenci = ogrenci_getir(ogrenci_id)

    if request.method == "POST":
        veri, hata = form_dogrula(request.form)

        if hata:
            flash(hata)
        else:
            isim, sinif, python_puani, matematik_puani = veri

            db = get_db()

            db.execute("""
            UPDATE Ogrenciler
            SET
                Isim = ?,
                Sinif = ?,
                PythonPuani = ?,
                MatematikPuani = ?
            WHERE Id = ?
            """, (
                isim,
                sinif,
                python_puani,
                matematik_puani,
                ogrenci_id
            ))

            db.commit()

            flash("Öğrenci güncellendi.")

            return redirect(
                url_for(
                    "ogrenci_detay",
                    ogrenci_id=ogrenci_id
                )
            )

    return render_template(
        "ogrenci_duzenle.html",
        ogrenci=ogrenci
    )

@app.post("/ogrenci/<int:ogrenci_id>/sil")
def ogrenci_sil(ogrenci_id):
    ogrenci_getir(ogrenci_id)

    db = get_db()

    db.execute(
        "DELETE FROM Ogrenciler WHERE Id = ?",
        (ogrenci_id,)
    )

    db.commit()

    flash("Öğrenci silindi.")

    return redirect(
        url_for("ogrenci_listesi")
    )

@app.route("/ara")
def ogrenci_ara():
    kelime = request.args.get("q", "").strip()

    ogrenciler = get_db().execute("""
    SELECT
        Id,
        Isim,
        Sinif,
        PythonPuani,
        MatematikPuani,
        (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
    FROM Ogrenciler
    WHERE Isim LIKE ?
    ORDER BY Isim
    """, (f"%{kelime}%",)).fetchall()

    return render_template(
        "ogrenciler.html",
        ogrenciler=ogrenciler,
        arama=kelime
    )

@app.errorhandler(404)
def sayfa_bulunamadi(hata):
    return render_template("404.html"), 404

veritabani_hazirla()


# 49. Uygulamayı Çalıştırmak

Proje klasöründe:

```text
flask --app app run --debug
```

çalıştırın.

Tarayıcıdan genellikle:

```text
http://127.0.0.1:5000
```

adresine gidin.

İlk aşamada şu işlemleri test edin:

1. öğrenci listesi açılıyor mu?
2. yeni kayıt ekleniyor mu?
3. detay sayfası açılıyor mu?
4. düzenleme çalışıyor mu?
5. silme çalışıyor mu?
6. arama çalışıyor mu?
7. veritabanı uygulama yeniden başlasa da kayıtları koruyor mu?


# 50. Uygulama Akışını Test Etmek

CRUD uygulamasını geliştirirken her şeyi en sonda test etmek yerine adım adım ilerlemek daha doğrudur.

Önerilen sıra:

```text
1. Veritabanı
2. SELECT
3. Liste sayfası
4. INSERT
5. Ekleme formu
6. Detay sayfası
7. UPDATE
8. Düzenleme formu
9. DELETE
10. Arama
11. Hata kontrolleri
```

Bir aşama düzgün çalışmadan sonraki aşamaya geçmemek hata bulmayı kolaylaştırır.


# 51. Hata Ayıklama: Route Adlarını Kontrol Etmek

Şu hata görülebilir:

```text
BuildError
```

Bunun nedenlerinden biri `url_for()` içinde yanlış endpoint adı kullanmaktır.

Örneğin route fonksiyonu:

```python
def ogrenci_listesi():
```

ise:

```python
url_for("ogrenci_listesi")
```

kullanılmalıdır.


# 52. Hata Ayıklama: Template Bulunamadı

Şu hata:

```text
TemplateNotFound
```

oluşursa kontrol edin:

```text
templates/
    index.html
```

Flask template dosyalarını varsayılan olarak `templates` klasöründe arar.


# 53. Hata Ayıklama: Veritabanı Tablosu Yok

Şu hata görülebilir:

```text
no such table: Ogrenciler
```

Bu durumda:

- doğru `.db` dosyasına bağlandığınızı,
- tablo hazırlama fonksiyonunun çalıştığını,
- uygulamanın hangi klasörden çalıştırıldığını

kontrol edin.


# 54. Hata Ayıklama: Form Alan Adları

HTML:

```html
<input name="python">
```

ise Python:

```python
request.form.get("python")
```

olmalıdır.

`name` değerleri ile Flask tarafındaki anahtarların aynı olması gerekir.


# 55. Hata Ayıklama: `commit()` Unutmak

`INSERT`, `UPDATE` veya `DELETE` yaptıktan sonra:

```python
db.commit()
```

kullanılmalıdır.

Aksi halde değişiklikler kalıcı olmayabilir.


# 56. Flask ile SQLite Kullanırken Bağlantı Yönetimi

Bu projede veritabanı bağlantısını:

```python
g
```

üzerinde istek süresince tutuyoruz.

İstek sonunda:

```python
@app.teardown_appcontext
```

ile bağlantıyı kapatıyoruz.

Bu yapı her route içinde ayrı ayrı bağlantı açıp kapatma tekrarını azaltır.


# 57. İleri Geliştirme: CSV Dışa Aktarma

Bir sonraki geliştirme olarak öğrenci listesini CSV dosyasına dönüştürebiliriz.

Burada önce SQLite verisini Pandas'a aktarabiliriz:

```python
import pandas as pd

df = pd.read_sql_query(
    "SELECT * FROM Ogrenciler",
    get_db()
)
```

Ardından CSV üretilebilir.

Bu özellik veri analizi derslerimizle web uygulamasını birleştirir.


# 58. İleri Geliştirme: Grafik Sayfası

SQLite verisini Pandas ve Matplotlib ile analiz edip grafik oluşturabiliriz.

Akış:

```text
SQLite
↓
Pandas
↓
Matplotlib
↓
PNG grafik
↓
Flask static klasörü
↓
HTML
```

Böylece web uygulaması yalnızca kayıt sistemi değil, raporlama uygulaması haline gelir.


# 59. İleri Geliştirme: Kullanıcı Giriş Sistemi

Gerçek uygulamalarda bazı işlemler yalnızca yetkili kullanıcılar tarafından yapılabilir.

Örneğin:

- öğretmen giriş yapar,
- öğrenci kayıtlarını yönetir,
- normal ziyaretçi yalnızca listeyi görebilir.

Bunun için ileride:

- session,
- kullanıcı tablosu,
- parola hashleme,
- login / logout,
- yetkilendirme

konuları kullanılabilir.


# 60. İleri Geliştirme: Uygulamayı Modüllere Ayırmak

Uygulama büyüdükçe tek `app.py` dosyası yetersiz hale gelir.

Örneğin:

```text
ogrenci_web/
│
├── app.py
├── db.py
├── ogrenciler.py
├── templates/
├── static/
└── instance/
    └── ogrenciler.db
```

Daha ileri Flask projelerinde:

- application factory,
- blueprint,
- instance klasörü

gibi yapılar kullanılabilir.


# 61. Masaüstü CRUD ile Web CRUD Arasındaki Bağlantı

Önceki Tkinter projemizde:

```text
Entry
Button
Treeview
SQLite
```

kullanmıştık.

Şimdi:

```text
HTML input
HTML form
HTML table
Flask route
SQLite
```

kullanıyoruz.

Arayüz değişmiştir, fakat CRUD ve veritabanı mantığı aynıdır.


# 62. Veri Akışını Baştan Sona Takip Edelim

Yeni öğrenci ekleme:

```text
Kullanıcı formu açar
↓
GET /ogrenci-ekle
↓
HTML form gösterilir
↓
Kullanıcı veriyi girer
↓
POST /ogrenci-ekle
↓
request.form
↓
form_dogrula()
↓
INSERT
↓
commit()
↓
flash()
↓
redirect()
↓
GET /ogrenciler
↓
SELECT
↓
Jinja
↓
HTML tablo
```

Bu akışı anlamak, tek tek komutları ezberlemekten daha önemlidir.


# 63. Ders Özeti

Bu derste:

- Flask + SQLite bağlantısı,
- `g`,
- `teardown_appcontext`,
- `sqlite3.Row`,
- veritabanı oluşturma,
- template yapısı,
- `base.html`,
- kayıt listeleme,
- detay sayfası,
- form doğrulama,
- `INSERT`,
- `SELECT`,
- `UPDATE`,
- `DELETE`,
- dinamik ID route,
- POST ile silme,
- `flash`,
- `redirect`,
- `url_for`,
- isimle arama,
- query string,
- sınıf filtreleme,
- sıralama,
- istatistik sorguları,
- `GROUP BY`,
- 404 hata sayfası,
- SQL Injection farkındalığı,
- HTML escaping farkındalığı,
- hata ayıklama,
- CRUD test süreci

konularını gerçek bir web projesinde birleştirdik.


# 64. Mini Uygulamalar

1. Öğrenci tablosuna `OgrenciNo` alanı ekleyin.
2. `OgrenciNo` alanını `UNIQUE` yapın.
3. Aynı öğrenci numarası girildiğinde kullanıcıya hata gösterin.
4. Öğrenci listesine öğrenci numarası sütunu ekleyin.
5. Öğrenci numarası ile arama ekleyin.
6. 7. sınıf öğrencilerini filtreleyen sayfa oluşturun.
7. 8. sınıf öğrencilerini filtreleyen sayfa oluşturun.
8. Python puanına göre sıralama route'u oluşturun.
9. Matematik puanına göre sıralama route'u oluşturun.
10. Ortalama 80 ve üzeri öğrencileri filtreleyin.
11. Öğrenci detay sayfasına başarı durumu ekleyin.
12. Toplam öğrenci sayısını ana sayfada gösterin.
13. Genel ortalamayı ana sayfada gösterin.
14. Sınıflara göre ortalama tablosu oluşturun.
15. Formda sınıfı select kutusuna dönüştürün.
16. Başarılı kayıt sonrası kategori içeren flash mesajı gösterin.
17. Silme işlemi için ayrı onay sayfası oluşturun.
18. Liste tablosuna sayfa bulunamadığında uygun mesaj ekleyin.
19. CSS ile form ve tablo görünümünü düzenleyin.
20. CSV dışa aktarma özelliği planlayın.
21. Pandas ile veritabanı raporu üretin.
22. Matplotlib ile başarı grafiği oluşturun.
23. Uygulamayı `db.py` ve `app.py` olarak bölün.
24. Öğrenci sistemi yerine aynı CRUD mantığıyla stok sistemi geliştirin.
25. Baştan sona kendi Flask + SQLite CRUD projenizi hazırlayın.


# 65. Proje Görevi

Bir **BİLSEM Proje Takip Web Uygulaması** geliştirin.

Her proje için:

- proje ID,
- proje adı,
- öğrenci,
- danışman,
- kategori,
- durum,
- puan

bilgileri tutulabilir.

Uygulamada en az:

- SQLite,
- Flask,
- `base.html`,
- static CSS,
- liste sayfası,
- detay sayfası,
- ekleme formu,
- düzenleme formu,
- silme işlemi,
- arama,
- filtreleme,
- form doğrulama,
- flash mesajları,
- özel 404 sayfası,
- istatistik sayfası

bulunsun.

CRUD işlemlerinin tamamı çalışmalıdır.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki tam web uygulaması zincirini kurabilmesi hedeflenmektedir:

**Kullanıcı**

↓

**HTML Form / Link**

↓

**HTTP GET veya POST**

↓

**Flask Route**

↓

**Python İş Mantığı**

↓

**SQL CRUD**

↓

**SQLite**

↓

**Jinja Template**

↓

**HTML Cevabı**

Bu noktada öğrenciler:

- masaüstü CRUD,
- web CRUD,
- veritabanı,
- veri analizi,
- grafik

gibi farklı alanların birbirine nasıl bağlandığını görmüş durumdadır.

Bir sonraki aşamada web uygulamasını daha profesyonel hale getirmek için **kullanıcı girişi, session, parola güvenliği ve yetkilendirme** konularına geçebiliriz.
